# 07 — Tiered Memory Architecture (Issue 9)

Demonstrates the tiered memory system that manages LLM context across simulation days:
1. Load agents with accumulated reflections (from prior phases)
2. Compress daily memories → daily summaries (2–3 sentences)
3. Compress weekly memories → weekly summaries (2–3 sentences)
4. Assemble full context and inspect structure at different day offsets
5. Verify persona drift mitigation (reinforcement every N days)

**Covers:** Issue 9 (Tiered Memory Architecture)  
**Depends on:** Issue 6

In [ ]:
import os, sys, random
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import ClimatePolicyID, SURVEY_QUESTIONS
from cag.io.llm import load_api_key

# Attribute maps and IDs
from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

print("Imports OK")

## 1. Load Data & Build Environment

In [ ]:
random.seed(42)
year = 2026

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData_train.csv")

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
sn.assign_political_exposure()
sn.create_network(seed=42)
sn.assign_network_blocks()

api_key = load_api_key("../data/api_key.csv")
target_policy = ClimatePolicyID.CARBON_TAX

print(f"Total citizens: {len(sn.agents_active)}")

## 2. Inject Synthetic Reflections (Multi-Day)

To test memory compression without running expensive LLM phases,
inject mock reflections spanning several days for a single demo citizen.

In [ ]:
# TODO: Inject mock reflections for days 1-10 on a single citizen
# citizen.reflections = [
#     {"day": d, "phase": phase, "text": f"...", "messages_received": ["..."]}
#     for d in range(1, 11) for phase in ["P-A", "P-B", "C"]
# ]

## 3. Daily Memory Compression

Compress reflections from day d-2 into a 2–3 sentence daily summary.

In [ ]:
# TODO: Call compress_daily_memory() for days 3+
# Show the summary text and verify it's 2-3 sentences

## 4. Weekly Memory Compression

Compress daily summaries from week 1 (days 1–7) into a weekly summary.

In [ ]:
# TODO: Call compress_weekly_memory() for week 1
# Show the weekly summary text

## 5. Context Assembly at Different Days

Inspect what `assemble_context(day)` produces at various simulation days:
- Day 1: persona + current reflections only
- Day 3: persona + full reflections (days 2–3)
- Day 5: persona + daily summary (day 3) + full reflections (days 4–5)
- Day 10: persona + weekly summary (week 1) + daily summaries + recent reflections

In [ ]:
# TODO: Print assemble_context(day) for days 1, 3, 5, 10
# Show the full context string with section labels

## 6. Opinion Trajectory in Context

In [ ]:
# TODO: Verify opinion trajectory string appears in context
# e.g., "Day 0: C, Day 1: D, Day 2: D, ..."

## 7. Persona Drift Mitigation

Verify persona reinforcement appears at end of context every N days.

In [ ]:
# TODO: Check persona reinforcement at day 5 and day 10
# Verify key persona attributes are repeated at end of context

## 8. Context Token Length Analysis

Track how context length grows with and without compression.

In [ ]:
# TODO: Compare context word/token counts across days
# - Without compression: raw reflections accumulate
# - With compression: bounded by tiered summaries

## 9. Sanity Checks

In [ ]:
# TODO: Sanity checks
# - Daily summaries exist for compressed days
# - Weekly summaries exist for completed weeks
# - Context at day 1 contains no summaries, just persona + reflections
# - Context grows sub-linearly with number of days
# - Persona reinforcement interval is respected